# Serving-stack Nondeterminism - R11-H229 (+ R22 gates H232/H233/H239)

**H229 (full test)** - how much of the temp-0 extraction variance is the serving stack, not the
model. Re-run one arm-A cell (5 serial runs x 3 documents, production prompt, temp 0, ONE request
at a time) on the shared vLLM `gpt-oss-120b`, mean pairwise Jaccard distance vs the same documents'
H119 concurrent arm-A values. Bar: serial <= 0.5x concurrent = CONFIRMED; serial >= 0.8x concurrent
= REFUTED.

**Three synthetic gates** (1 document each, 3 runs; a failed gate closes its hypothesis as
not-worth-testing):
- **H232 gate** - per-request `seed` pinned, serial. Gate: JD < 0.3 on the doc. Also record the
  exact byte-identical reproduction rate
- **H233 gate** - guided/structured decoding (vLLM `response_format` json_schema on the extraction
  schema). Gate: >= 15% JD reduction vs the doc's free-run JD
- **H239 gate** - a second local extractor (Qwen2.5-7B-Instruct Q4_K_M, llama.cpp, GPU 0). Gate:
  single-doc variance floor within 1.5x of gpt-oss-120b's on the same doc -> close as
  not-worth-testing; outside 1.5x -> full test earns its slot

**Ambient load (honest note)** - the vLLM server is shared: H119 is done but a light single-doc
paper ingest (H157) still runs against `gpt-oss-120b` on GPU 1. These serial runs are NOT on a
perfectly idle server; per-run wall-clock is recorded and the residual concurrency is noted with the
result. GPU 1 is never touched; the second model is stood up on GPU 0.

## GPU selection

The gpt-oss-120b arms are CLIENT-ONLY against the already-running vLLM server (port 8010, GPU 1) -
this notebook starts no GPU work for them and imports no torch/tensorflow, so no
`CUDA_VISIBLE_DEVICES` is needed here. The H239 second model (Qwen2.5-7B) is served by an external
`llama-server` process pinned to GPU 0 via `CUDA_VISIBLE_DEVICES=0` at launch (outside this
notebook); the notebook is a client to it on port 8011.

In [1]:
# Imports - grouped by category
from __future__ import annotations
import os                                    # cwd normalization under nbconvert
import json                                  # report + checkpoint serialization
import pickle                                # chunk cache
import time                                  # per-run timing
import itertools                             # run-pair enumeration
import sys                                   # loguru sink
from pathlib import Path                     # filesystem paths
from datetime import datetime, timezone      # UTC report timestamp
from statistics import mean                  # metric aggregation

from loguru import logger                    # engine logging (quieted below)

# Project modules (extraction engine, prompts, chunking, models) - identical to the H119 harness
from knowledge_graph_foundry.extraction.extractor import extract_document, WireExtraction
from knowledge_graph_foundry.settings import LLMSettings, ExtractionSettings
from knowledge_graph_foundry.engines.local_gpu import LocalGpuEngine
from knowledge_graph_foundry.engines.base import EngineError
from knowledge_graph_foundry.models import Chunk, Ontology, normalize_name

logger.remove()                              # silence per-chunk DEBUG spam
logger.add(sys.stderr, level="WARNING")

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("imports ok; cwd:", Path.cwd())

2026-07-08 09:58:04.795 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


imports ok; cwd: /home/lab/workspace/learning/projects/knowledge-graph-foundry


## Configuration

Endpoints, the deterministic 3-document selection (first 3 of the H119 sorted set), the pinned seed,
the H119 concurrent arm-A reference values, and output paths are frozen here before any run. The
extraction settings mirror production exactly (`split_entity_relation=True`, `gleaning_rounds=1`,
chunk 2000/overlap 200), identical to H119. Serial submission means `concurrency=1` inside
`extract_document` AND documents processed one at a time - one request in flight at any moment.

In [2]:
# --- Frozen configuration ---
ENDPOINT_A  = "http://localhost:8010/v1"     # shared vLLM gpt-oss-120b (GPU 1)
MODEL_A     = "gpt-oss-120b"
ENDPOINT_B  = "http://127.0.0.1:8011/v1"     # H239 second model, llama.cpp (GPU 0)
MODEL_B     = "models/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf"
PURPOSE     = "compare CPAP machines"
TEMPERATURE = 0.0
SEED        = 1234                            # H232 per-request pinned seed
N_RUNS_H229 = 5                             # H229 full test
N_RUNS_GATE = 3                             # each synthetic gate
GUIDED_MAX_TOKENS = 16000                    # reasoning headroom for grammar-constrained gpt-oss

CORPUS_DIR  = Path("data/external/cpap-datasheets-and-manuals")
CHUNK_CACHE = Path("data/interim/h119_chunks.pkl")
CKPT_DIR    = Path("results/h229")
LOG_PATH    = Path("logs/h229-gates.log")
REPORTS_DIR = Path("reports")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Deterministic document selection: sorted order, first 3 (matches H119's sorted set).
DOCS = [p.name for p in sorted(CORPUS_DIR.glob("*.pdf"))][:3]
GATE_DOC = DOCS[0]                            # the single-doc gate document

# H119 concurrent arm-A per-doc mean pairwise Jaccard distance (frozen from its report).
H119_REPORT = Path("reports/extraction-determinism-h119-20260708T010118Z.json")
_h119 = json.loads(H119_REPORT.read_text())
CONCURRENT_A = {d: _h119["per_doc_distance"]["A_production"][d] for d in DOCS}
CONCURRENT_A_MEAN = mean(CONCURRENT_A.values())

FIXED_ONTOLOGY = Ontology()                  # held fixed (empty) across every run, as in H119
EXTRACTION_CFG = ExtractionSettings()        # chunk 2000/200, glean 1, split True

def log_line(msg: str) -> None:
    stamp = datetime.now(timezone.utc).strftime("%H:%M:%S")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a") as fh:
        fh.write(line + "\n")

print("documents (first 3 of H119 sorted set):")
for d in DOCS:
    print(f"  - {d}   concurrent arm-A JD = {CONCURRENT_A[d]:.4f}")
print(f"concurrent arm-A mean over 3 docs = {CONCURRENT_A_MEAN:.4f}")
print(f"gate document = {GATE_DOC}")
print(f"H229 bars: CONFIRMED serial <= {0.5*CONCURRENT_A_MEAN:.4f}  |  REFUTED serial >= {0.8*CONCURRENT_A_MEAN:.4f}")

documents (first 3 of H119 sorted set):
  - 0-20190113114505.pdf   concurrent arm-A JD = 0.4000
  - 1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf   concurrent arm-A JD = 0.7787
  - 3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf   concurrent arm-A JD = 0.7735
concurrent arm-A mean over 3 docs = 0.6507
gate document = 0-20190113114505.pdf
H229 bars: CONFIRMED serial <= 0.3254  |  REFUTED serial >= 0.5206


## Frozen metric definitions

Identical to the H119 harness so the numbers are directly comparable: case/whitespace/hyphen name
fold, entity set per (doc, run) = normalized names, variance = mean pairwise Jaccard DISTANCE across
the runs (all unordered pairs), averaged over documents.

In [3]:
# --- Frozen metric functions (identical to H119) ---
def norm(name: str) -> str:
    base = normalize_name(name)                      # strip + lower + ws-collapse
    return " ".join(base.replace("-", " ").split())  # + hyphen fold

def jaccard_distance(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return 1.0 - len(a & b) / len(a | b)

def mean_pairwise_distance(sets: list) -> float:
    pairs = list(itertools.combinations(range(len(sets)), 2))
    return mean(jaccard_distance(sets[i], sets[j]) for i, j in pairs)

def set_of(names: list) -> set:
    return {norm(x) for x in names}

print("metric functions frozen")

metric functions frozen


## Data loading

Reuse the exact `h119_chunks.pkl` cache (parse-once chunks); take the first 3 documents. Identical
chunk text feeds every run and arm, isolating extraction (LLM) variance from parse variance.

In [4]:
# Load cached chunks (must exist from H119).
assert CHUNK_CACHE.exists(), "h119_chunks.pkl missing - run H119 data loading first"
chunk_cache = pickle.loads(CHUNK_CACHE.read_bytes())
assert all(d in chunk_cache for d in DOCS), "cache missing a selected document"
CHUNKS = {name: [Chunk(**cd) for cd in chunk_cache[name]] for name in DOCS}
for name in DOCS:
    print(f"{len(CHUNKS[name]):2d} chunks  {name}")
print("total chunks (3 docs):", sum(len(v) for v in CHUNKS.values()))

 1 chunks  0-20190113114505.pdf
 2 chunks  1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf
 8 chunks  3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf
total chunks (3 docs): 11


## Engines - the three levers

All engines share the production extraction path (`extract_document`, split entity/relation prompts,
gleaning, temp 0). Only the decoding lever changes:

- **FreeEngine** - the production `LocalGpuEngine` unchanged (instructor JSON mode). Used for H229,
  the H233 free reference, and H239 (against the second model)
- **SeededEngine** - adds a pinned per-request `seed` to the instructor call (H232)
- **GuidedEngine** - bypasses instructor's prompt-injected JSON and uses vLLM grammar-constrained
  decoding via `response_format` json_schema built from the response model (H233)

The DEF-6 explicit `instructor.v2.providers.openai.handlers` import registers the `(OPENAI, Mode.JSON)`
handler; construction is serialized under a lock exactly as H119 does.

In [5]:
# instructor v2 mode registry is populated by import side effects (DEF-6): register explicitly.
import instructor.v2.providers.openai.handlers  # noqa: F401  registers (OPENAI, Mode.JSON)
import litellm
import threading
litellm.suppress_debug_info = True
_engine_lock = threading.Lock()

class SeededEngine(LocalGpuEngine):
    '''LocalGpuEngine + pinned per-request seed (H232).'''
    def __init__(self, cfg, seed):
        super().__init__(cfg)
        self._seed = seed
    def complete(self, messages, response_model):
        model = self.cfg.model
        if not model.startswith("openai/"):
            model = f"openai/{model}"
        try:
            return self._client.chat.completions.create(
                model=model, messages=messages, response_model=response_model,
                temperature=self.cfg.temperature, timeout=self.cfg.timeout,
                max_retries=self.cfg.max_retries, api_base=self.cfg.base_url,
                api_key="local", seed=self._seed)
        except Exception as exc:
            raise EngineError(f"seeded engine failed: {exc}") from exc

class GuidedEngine(LocalGpuEngine):
    '''vLLM grammar-constrained decoding via response_format json_schema (H233).'''
    def __init__(self, cfg, max_tokens):
        super().__init__(cfg)
        self._max_tokens = max_tokens
    def complete(self, messages, response_model):
        model = self.cfg.model
        if not model.startswith("openai/"):
            model = f"openai/{model}"
        schema = response_model.model_json_schema()
        try:
            resp = litellm.completion(
                model=model, messages=messages, temperature=self.cfg.temperature,
                timeout=self.cfg.timeout, api_base=self.cfg.base_url, api_key="local",
                max_tokens=self._max_tokens,
                response_format={"type": "json_schema",
                                 "json_schema": {"name": response_model.__name__, "schema": schema}})
            choice = resp.choices[0]
            if choice.finish_reason == "length":
                raise EngineError("guided decode truncated (finish_reason=length)")
            return response_model.model_validate_json(choice.message.content)
        except EngineError:
            raise
        except Exception as exc:
            raise EngineError(f"guided engine failed: {exc}") from exc

def make_free(endpoint, model):
    cfg = LLMSettings(engine="local-gpu", model=model, base_url=endpoint,
                      temperature=TEMPERATURE, timeout=300)
    with _engine_lock:
        return LocalGpuEngine(cfg)

def make_seeded(endpoint, model, seed):
    cfg = LLMSettings(engine="local-gpu", model=model, base_url=endpoint,
                      temperature=TEMPERATURE, timeout=300)
    with _engine_lock:
        return SeededEngine(cfg, seed)

def make_guided(endpoint, model, max_tokens):
    cfg = LLMSettings(engine="local-gpu", model=model, base_url=endpoint,
                      temperature=TEMPERATURE, timeout=300)
    with _engine_lock:
        return GuidedEngine(cfg, max_tokens)

# warm the engine construction path once (matches H119)
_ = make_free(ENDPOINT_A, MODEL_A)
print("engines wired (free / seeded / guided)")

engines wired (free / seeded / guided)


## Serial extraction harness

One request at a time: `extract_document(..., concurrency=1)` per document, documents looped
sequentially. Every (tag, run, doc) result is checkpointed to `results/h229/` so re-execution
resumes rather than repeats LLM work. Per-run wall-clock is logged - the honest ambient-load signal.

In [6]:
def ckpt_path(tag, run, docname):
    safe = docname.replace("/", "_")
    return CKPT_DIR / f"{tag}__run{run}__{safe}.json"

def extract_serial(engine, docname):
    '''Raw entity names from one document, ONE request at a time (concurrency=1).'''
    result = extract_document(CHUNKS[docname], PURPOSE, FIXED_ONTOLOGY, engine,
                              concurrency=1, extraction_cfg=EXTRACTION_CFG)
    return [e.name for e in result.entities]

def run_serial(tag, engine_factory, docnames, n_runs):
    '''Serial runs x docs. engine_factory() -> fresh engine per (run, doc). Returns
    {run: {doc: names}}. Resumable via checkpoints.'''
    out = {r: {} for r in range(1, n_runs + 1)}
    log_line(f"=== {tag}: {n_runs} serial runs x {len(docnames)} doc(s) ===")
    for run in range(1, n_runs + 1):
        for docname in docnames:
            cp = ckpt_path(tag, run, docname)
            if cp.exists():
                out[run][docname] = json.loads(cp.read_text())["names"]
                continue
            eng = engine_factory()
            t0 = time.time()
            names = extract_serial(eng, docname)
            dt = time.time() - t0
            cp.write_text(json.dumps({"tag": tag, "run": run, "doc": docname,
                                      "names": names, "count": len(names),
                                      "seconds": round(dt, 1)}))
            out[run][docname] = names
            log_line(f"{tag} run{run} {docname[:42]:42s} n={len(names):3d} {dt:6.1f}s")
    return out

print("serial harness ready")

serial harness ready


## Task 1 - H229: serial 5x3, free production prompt, temp 0

Five serial runs over the three documents on the shared gpt-oss-120b, one request at a time.

In [7]:
H229 = run_serial("H229_free", lambda: make_free(ENDPOINT_A, MODEL_A), DOCS, N_RUNS_H229)

per_doc_serial = {}
for d in DOCS:
    sets = [set_of(H229[r][d]) for r in range(1, N_RUNS_H229 + 1)]
    per_doc_serial[d] = mean_pairwise_distance(sets)
serial_mean = mean(per_doc_serial.values())

ratio = serial_mean / CONCURRENT_A_MEAN
if ratio <= 0.5:
    h229_verdict = "CONFIRMED"
elif ratio >= 0.8:
    h229_verdict = "REFUTED"
else:
    h229_verdict = "INDETERMINATE"

print("H229 per-doc serial vs concurrent arm-A:")
for d in DOCS:
    print(f"  {d[:44]:44s} serial={per_doc_serial[d]:.4f}  concurrent={CONCURRENT_A[d]:.4f}")
print(f"serial mean     = {serial_mean:.4f}")
print(f"concurrent mean = {CONCURRENT_A_MEAN:.4f}")
print(f"serial / concurrent ratio = {ratio:.3f}")
print(f"H229 verdict = {h229_verdict}  (CONFIRMED<=0.5, REFUTED>=0.8)")

[07:58:06] === H229_free: 5 serial runs x 3 doc(s) ===
H229 per-doc serial vs concurrent arm-A:
  0-20190113114505.pdf                         serial=0.7086  concurrent=0.4000
  1017900r4_ResMed_Product_Catalogue_ANZ_Eng_L serial=0.4289  concurrent=0.7787
  3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1 serial=0.6658  concurrent=0.7735
serial mean     = 0.6011
concurrent mean = 0.6507
serial / concurrent ratio = 0.924
H229 verdict = REFUTED  (CONFIRMED<=0.5, REFUTED>=0.8)


## Task 2 - H232 gate: seed pinned, serial, 1 document x 3 runs

Per-request seed pinned to a constant, three serial runs on the gate document. Gate bar: JD < 0.3.
Also record the exact byte-identical reproduction rate - the fraction of run-pairs whose normalized
entity sets are identical, and whether the raw ordered name lists are byte-identical.

In [8]:
H232 = run_serial("H232_seed", lambda: make_seeded(ENDPOINT_A, MODEL_A, SEED), [GATE_DOC], N_RUNS_GATE)

h232_sets = [set_of(H232[r][GATE_DOC]) for r in range(1, N_RUNS_GATE + 1)]
h232_jd = mean_pairwise_distance(h232_sets)

# exact reproduction: normalized-set identity and raw ordered-list identity, over run-pairs
h232_raw = [H232[r][GATE_DOC] for r in range(1, N_RUNS_GATE + 1)]
pairs = list(itertools.combinations(range(N_RUNS_GATE), 2))
set_identical = sum(1 for i, j in pairs if h232_sets[i] == h232_sets[j])
raw_identical = sum(1 for i, j in pairs if h232_raw[i] == h232_raw[j])
h232_set_repro = set_identical / len(pairs)
h232_raw_repro = raw_identical / len(pairs)
h232_all_byte_identical = all(h232_raw[0] == h232_raw[k] for k in range(1, N_RUNS_GATE))

h232_gate = "GO" if h232_jd < 0.3 else "NO-GO"
print(f"H232 seeded-serial JD ({GATE_DOC[:40]}) = {h232_jd:.4f}   gate bar < 0.3")
print(f"  normalized-set reproduction rate = {h232_set_repro:.3f} ({set_identical}/{len(pairs)} pairs)")
print(f"  raw ordered-list reproduction rate = {h232_raw_repro:.3f} ({raw_identical}/{len(pairs)} pairs)")
print(f"  all 3 runs byte-identical = {h232_all_byte_identical}")
print(f"  entity counts per run = {[len(x) for x in h232_raw]}")
print(f"H232 GATE = {h232_gate}")

[07:58:06] === H232_seed: 3 serial runs x 1 doc(s) ===
H232 seeded-serial JD (0-20190113114505.pdf) = 0.5509   gate bar < 0.3
  normalized-set reproduction rate = 0.000 (0/3 pairs)
  raw ordered-list reproduction rate = 0.000 (0/3 pairs)
  all 3 runs byte-identical = False
  entity counts per run = [48, 40, 36]
H232 GATE = NO-GO


## Task 3 - H233 gate: guided decoding, 1 document x 3 runs

Grammar-constrained decoding (vLLM `response_format` json_schema on the extraction schema), three
serial runs on the gate document. Gate bar: >= 15% JD reduction vs the same document's free-run JD.
The free reference is the gate document's H229 serial free runs (3-run subset for an apples-to-apples
3-vs-3 comparison; the 5-run value is reported too).

In [9]:
H233 = run_serial("H233_guided", lambda: make_guided(ENDPOINT_A, MODEL_A, GUIDED_MAX_TOKENS),
                   [GATE_DOC], N_RUNS_GATE)

h233_sets = [set_of(H233[r][GATE_DOC]) for r in range(1, N_RUNS_GATE + 1)]
h233_jd = mean_pairwise_distance(h233_sets)

# free reference from H229 on the same doc: 3-run subset (matched) and full 5-run
free_sets_all = [set_of(H229[r][GATE_DOC]) for r in range(1, N_RUNS_H229 + 1)]
free_jd_3 = mean_pairwise_distance(free_sets_all[:3])
free_jd_5 = mean_pairwise_distance(free_sets_all)

h233_reduction = (free_jd_3 - h233_jd) / free_jd_3 if free_jd_3 else 0.0
h233_gate = "GO" if h233_reduction >= 0.15 else "NO-GO"
print(f"H233 guided JD ({GATE_DOC[:40]}) = {h233_jd:.4f}")
print(f"  free-run JD (3-run matched) = {free_jd_3:.4f}   (5-run = {free_jd_5:.4f})")
print(f"  JD reduction vs free (3-run) = {h233_reduction:.3f}   gate bar >= 0.15")
print(f"  guided entity counts per run = {[len(x) for x in h233_sets]}")
print(f"  free   entity counts per run = {[len(x) for x in free_sets_all[:3]]}")
print(f"H233 GATE = {h233_gate}")

[07:58:07] === H233_guided: 3 serial runs x 1 doc(s) ===
H233 guided JD (0-20190113114505.pdf) = 0.8545
  free-run JD (3-run matched) = 0.7047   (5-run = 0.7086)
  JD reduction vs free (3-run) = -0.213   gate bar >= 0.15
  guided entity counts per run = [41, 39, 50]
  free   entity counts per run = [47, 46, 20]
H233 GATE = NO-GO


## Task 4 - H239 gate: second model, 1 document x 3 runs

Qwen2.5-7B-Instruct Q4_K_M served by llama.cpp on GPU 0, same prompt, temp 0, serial.

**Deviation (recorded honestly)**: the free production decode path FAILED on Qwen-7B - the model
ignores the JSON instruction on the production entity prompt and emits a raw token list; instructor
exhausted 4 attempts per chunk and `extract_document` returned empty results. A 7B-class model
cannot run the production path unmodified (itself a finding). To measure its variance floor the gate
reruns with llama.cpp grammar-constrained decoding (`response_format` json_schema - the same
GuidedEngine as H233), and the 1.5x band is evaluated against BOTH gpt-oss references on the same
document: the free-decode JD (primary, per registration) and the guided-decode JD (matched decode
mode, from H233).

Gate: within 1.5x of gpt-oss-120b's floor -> close H239 as not-worth-testing; outside 1.5x -> the
full test earns its slot.

In [10]:
# health check the second server before running
import urllib.request
try:
    with urllib.request.urlopen(ENDPOINT_B.replace('/v1','') + "/health", timeout=10) as r:
        print("second model health:", r.read().decode()[:60])
except Exception as e:
    print("second model health check failed:", e)

# Free-decode attempt (H239_qwen tag) produced n=0 on all runs - instructor json_invalid, see
# deviation note above. The gate runs grammar-constrained (H239_qwen_guided tag).
H239 = run_serial("H239_qwen_guided",
                  lambda: make_guided(ENDPOINT_B, MODEL_B, GUIDED_MAX_TOKENS),
                  [GATE_DOC], N_RUNS_GATE)

h239_sets = [set_of(H239[r][GATE_DOC]) for r in range(1, N_RUNS_GATE + 1)]
h239_jd = mean_pairwise_distance(h239_sets)
gpt_free_jd = free_jd_3    # gpt-oss free-decode reference (registration's primary)
gpt_guided_jd = h233_jd    # gpt-oss guided-decode reference (matched decode mode)

def band_ratio(x, ref):
    if ref > 0:
        return x / ref
    return float("inf") if x > 0 else 1.0

ratio_vs_free = band_ratio(h239_jd, gpt_free_jd)
ratio_vs_guided = band_ratio(h239_jd, gpt_guided_jd)
within_free = (1/1.5) <= ratio_vs_free <= 1.5
within_guided = (1/1.5) <= ratio_vs_guided <= 1.5
# primary decision on the registration's reference (free); matched-mode reported alongside
h239_gate = "NO-GO" if within_free else "GO"   # NO-GO = close as not-worth-testing
print(f"H239 Qwen-7B guided single-doc JD ({GATE_DOC[:40]}) = {h239_jd:.4f}")
print(f"  gpt-oss free JD (same doc, 3-run)   = {gpt_free_jd:.4f}  ratio={ratio_vs_free:.3f}  within1.5x={within_free}")
print(f"  gpt-oss guided JD (same doc, 3-run) = {gpt_guided_jd:.4f}  ratio={ratio_vs_guided:.3f}  within1.5x={within_guided}")
print(f"  qwen entity counts per run = {[len(x) for x in h239_sets]}")
print(f"H239 GATE = {h239_gate}   (primary vs free reference; NO-GO = within 1.5x -> close not-worth-testing)")

second model health: {"status":"ok"}
[07:58:07] === H239_qwen_guided: 3 serial runs x 1 doc(s) ===


[07:58:27] H239_qwen_guided run1 0-20190113114505.pdf                       n= 17   20.0s


[07:58:48] H239_qwen_guided run2 0-20190113114505.pdf                       n= 20   21.5s


[07:59:23] H239_qwen_guided run3 0-20190113114505.pdf                       n= 20   34.5s
H239 Qwen-7B guided single-doc JD (0-20190113114505.pdf) = 0.4198
  gpt-oss free JD (same doc, 3-run)   = 0.7047  ratio=0.596  within1.5x=False
  gpt-oss guided JD (same doc, 3-run) = 0.8545  ratio=0.491  within1.5x=False
  qwen entity counts per run = [17, 20, 20]
H239 GATE = GO   (primary vs free reference; NO-GO = within 1.5x -> close not-worth-testing)


## Report

Machine-readable JSON with per-task numbers and GO/NO-GO decisions, written to `reports/`.

In [11]:
utc = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {
    "hypothesis": "R11-H229 (+ R22 gates H232/H233/H239)",
    "utc": utc,
    "endpoint_A": ENDPOINT_A, "model_A": MODEL_A,
    "endpoint_B": ENDPOINT_B, "model_B": MODEL_B,
    "purpose": PURPOSE, "temperature": TEMPERATURE, "seed": SEED,
    "documents": DOCS, "gate_document": GATE_DOC,
    "chunk_counts": {d: len(CHUNKS[d]) for d in DOCS},
    "ambient_load_note": ("shared vLLM gpt-oss-120b on GPU 1; H119 done but a light single-doc "
                          "H157 paper ingest still shares the server - runs are serial from the "
                          "client but the server is not perfectly idle; per-run wall-clock logged"),
    "h229": {
        "runs_per_doc": N_RUNS_H229,
        "per_doc_serial": per_doc_serial,
        "per_doc_concurrent_A": CONCURRENT_A,
        "serial_mean": serial_mean,
        "concurrent_mean": CONCURRENT_A_MEAN,
        "serial_over_concurrent_ratio": ratio,
        "bar": {"confirmed_at_or_below": 0.5, "refuted_at_or_above": 0.8},
        "verdict": h229_verdict,
    },
    "h232_gate": {
        "runs": N_RUNS_GATE, "seeded_serial_jd": h232_jd,
        "normalized_set_reproduction_rate": h232_set_repro,
        "raw_list_reproduction_rate": h232_raw_repro,
        "all_runs_byte_identical": h232_all_byte_identical,
        "entity_counts": [len(x) for x in h232_raw],
        "gate_bar": "JD < 0.3", "decision": h232_gate,
    },
    "h233_gate": {
        "runs": N_RUNS_GATE, "guided_jd": h233_jd,
        "free_jd_3run": free_jd_3, "free_jd_5run": free_jd_5,
        "jd_reduction_vs_free": h233_reduction,
        "guided_entity_counts": [len(x) for x in h233_sets],
        "gate_bar": ">= 15% JD reduction", "decision": h233_gate,
    },
    "h239_gate": {
        "runs": N_RUNS_GATE, "model": MODEL_B,
        "deviation": ("free production decode failed on Qwen-7B (non-JSON output, instructor "
                      "retries exhausted, n=0 across 3 runs) - gate rerun grammar-constrained "
                      "(llama.cpp response_format json_schema); band checked vs gpt-oss free "
                      "(primary) and guided (matched-mode) references"),
        "qwen_guided_jd": h239_jd,
        "gpt_oss_free_jd": gpt_free_jd, "ratio_vs_free": ratio_vs_free,
        "within_1_5x_vs_free": within_free,
        "gpt_oss_guided_jd": gpt_guided_jd, "ratio_vs_guided": ratio_vs_guided,
        "within_1_5x_vs_guided": within_guided,
        "qwen_entity_counts": [len(x) for x in h239_sets],
        "gate_bar": "within 1.5x -> close not-worth-testing", "decision": h239_gate,
    },
}
out_path = REPORTS_DIR / f"serving-determinism-h229-{utc}.json"
out_path.write_text(json.dumps(report, indent=2))
log_line(f"report written: {out_path}")
print("report:", out_path)
print(json.dumps({k: report[k] for k in ["h229", "h232_gate", "h233_gate", "h239_gate"]}, indent=2))

[07:59:23] report written: reports/serving-determinism-h229-20260708T075923Z.json
report: reports/serving-determinism-h229-20260708T075923Z.json
{
  "h229": {
    "runs_per_doc": 5,
    "per_doc_serial": {
      "0-20190113114505.pdf": 0.7085908634094541,
      "1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf": 0.4288956297730525,
      "3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf": 0.6657923350384791
    },
    "per_doc_concurrent_A": {
      "0-20190113114505.pdf": 0.4,
      "1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf": 0.7786909965034965,
      "3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf": 0.7734797750855512
    },
    "serial_mean": 0.6010929427403285,
    "concurrent_mean": 0.6507235905296825,
    "serial_over_concurrent_ratio": 0.9237300621774674,
    "bar": {
      "confirmed_at_or_below": 0.5,
      "refuted_at_or_above": 0.8
    },
    "verdict": "REFUTED"
  },
  "h232_gate": {
    "runs": 3,
    "seeded_serial_jd": 0.55093043559827